## 🥈 Silver Layer — 정제 및 표준화

Bronze의 원본 데이터를 **신뢰할 수 있는 데이터**로 가공하는 단계입니다.

- **타입 변환**: 문자열로 저장된 날짜/시간을 Timestamp로 변환
- **컬럼 정리**: 불필요한 컬럼 제거, 이름 표준화
- **파생 컬럼 추가**: 비즈니스 로직 기반의 새로운 컬럼 생성 (예: 배송 소요일)

> Bronze가 "있는 그대로"라면, Silver는 "믿을 수 있는" 데이터입니다.

In [0]:
# 라이브러리 불러오기
from pyspark.sql.functions import col, to_date, when, current_timestamp, datediff, to_timestamp, date_format, sum, count, avg

In [0]:
# 사용자 테이블 확인
customers = spark.table("brickstore.bronze.customers")
display(customers.limit(5))

In [0]:
# 스키마 생성
spark.sql("CREATE SCHEMA IF NOT EXISTS brickstore.silver")

In [0]:
# 데이터 정제
# - 데이터 타입 보정 - 날짜
# - 불필요한 데이터 삭제 - created_at
# - 프리미엄 멤버 플래그 추가
silver_customers = (customers
    .select(
        col("id").alias("customer_id"),
        col("name"),
        col("email"),
        col("phone"),
        col("grade"),
        col("region"),
        col("points"),
        col("total_spent"),
        to_date(col("signup_date")).alias("signup_date")
    )
    .withColumn("is_premium", when(col("grade").isin("VIP", "VVIP"), True).otherwise(False))
)

silver_customers.write.mode("overwrite").saveAsTable("brickstore.silver.customers")
display(silver_customers.limit(5))

In [0]:
# 주문 테이블 확인
orders = spark.table("brickstore.bronze.orders")
display(orders.limit(5))

In [0]:
# 데이터 정제
# - 데이터 타입 보정 - 타임스탬프
# - 배송 소요일 계산
silver_orders = (orders
    .select(
        col("id").alias("order_id"),
        col("customer_id"),
        col("order_group_id"),
        col("total_amount"),
        col("discount_amount"),
        col("coupon_code"),
        col("payment_method"),
        col("status"),
        to_timestamp(col("ordered_at")).alias("ordered_at"),
        to_timestamp(col("shipped_at")).alias("shipped_at"),
        to_timestamp(col("delivered_at")).alias("delivered_at"),
    )
    .withColumn(
        "delivery_days",
        when(col("delivered_at").isNotNull(),
             datediff(col("delivered_at"), col("ordered_at")))
    )
)

silver_orders.write.mode("overwrite").saveAsTable("brickstore.silver.orders")
display(silver_orders.limit(5))

In [0]:
# 주문 상품 테이블 확인
order_items = spark.table("brickstore.bronze.order_items")
display(order_items.limit(5))

In [0]:
# 상품 테이블 확인
products = spark.table("brickstore.bronze.products")
display(products.limit(5))

In [0]:
# 데이터 정제
# - 주문 상품에 상품 정보 추가
silver_order_items = (order_items
    .join(products, order_items.product_id == products.id, "left")
    .select(
        order_items["id"].alias("item_id"),
        order_items["order_id"],
        order_items["product_id"],
        products["name"].alias("product_name"),
        products["category"],
        order_items["quantity"],
        order_items["unit_price"],
        order_items["subtotal"],
        order_items["status"],
    )
)

silver_order_items.write.mode("overwrite").saveAsTable("brickstore.silver.order_items")
display(silver_order_items.limit(5))

In [0]:
# 반품 테이블 확인
returns = spark.table("brickstore.bronze.returns")
display(returns.limit(5))

In [0]:
# 데이터 정제
# - 반품 소요일 계산
silver_returns = (returns
    .select(
        col("id").alias("return_id"),
        col("order_id"),
        col("order_item_id"),
        col("reason"),
        col("status"),
        col("refund_amount"),
        to_timestamp(col("requested_at")).alias("requested_at"),
        to_timestamp(col("completed_at")).alias("completed_at"),
    )
    .withColumn(
        "processing_days",
        when(col("completed_at").isNotNull(),
             datediff(col("completed_at"), col("requested_at")))
    )
)

silver_returns.write.mode("overwrite").saveAsTable("brickstore.silver.returns")
display(silver_returns.limit(5))

## 🥇 Gold Layer — 집계 및 분석

Silver의 정제된 데이터를 **목적에 맞게 집계**하는 단계입니다.

- **비즈니스 메트릭**: 일별 매출, 고객별 구매 횟수 등 실제 의사결정에 필요한 지표
- **분석 최적화**: 대시보드, 리포트에서 바로 사용할 수 있는 형태
- **Materialized View**: 쿼리할 때마다 재계산하지 않고 결과를 미리 저장

> Silver가 "믿을 수 있는" 데이터라면, Gold는 "바로 쓸 수 있는" 데이터입니다.

In [0]:
# 스키마 생성
spark.sql("CREATE SCHEMA IF NOT EXISTS brickstore.gold")

In [0]:
# 월별 매출 요약
orders = spark.table("brickstore.silver.orders")

gold_monthly_sales = (orders
    .filter(col("status") != "취소")
    .withColumn("month", date_format(col("ordered_at"), "yyyy-MM"))
    .groupBy("month")
    .agg(
        count("order_id").alias("order_count"),
        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_order_value"),
        sum("discount_amount").alias("total_discount"),
    )
    .orderBy("month")
)

gold_monthly_sales.write.mode("overwrite").saveAsTable("brickstore.gold.monthly_sales")
display(gold_monthly_sales)

In [0]:
# 카테고리별 판매 분석
order_items = spark.table("brickstore.silver.order_items")

gold_category_sales = (order_items
    .filter(col("status") != "취소")
    .groupBy("category")
    .agg(
        count("item_id").alias("items_sold"),
        sum("subtotal").alias("total_revenue"),
        avg("unit_price").alias("avg_price"),
    )
    .orderBy(col("total_revenue").desc())
)

gold_category_sales.write.mode("overwrite").saveAsTable("brickstore.gold.category_sales")
display(gold_category_sales)


In [0]:
# 고객 등급별 분석
customers = spark.table("brickstore.silver.customers")
orders = spark.table("brickstore.silver.orders")

gold_grade_analysis = (orders
    .filter(col("status") != "취소")
    .join(customers, orders.customer_id == customers.customer_id)
    .groupBy("grade")
    .agg(
        count("order_id").alias("order_count"),
        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_order_value"),
    )
    .orderBy(col("total_revenue").desc())
)

gold_grade_analysis.write.mode("overwrite").saveAsTable("brickstore.gold.grade_analysis")
display(gold_grade_analysis)


In [0]:
# 반품 분석
returns = spark.table("brickstore.silver.returns")

gold_return_analysis = (returns
    .groupBy("reason")
    .agg(
        count("return_id").alias("return_count"),
        avg("processing_days").alias("avg_processing_days"),
        sum("refund_amount").alias("total_refund"),
    )
    .orderBy(col("return_count").desc())
)

gold_return_analysis.write.mode("overwrite").saveAsTable("brickstore.gold.return_analysis")
display(gold_return_analysis)